# Avance 5 — Modelo final de ensamble
**Proyecto Integrador · Equipo 54 · Fresas Donna y Dream**

Continuación reproducible del Avance 4. Ejecutar desde la carpeta que contiene los nueve archivos Excel RAW. Los resultados de ensambles se generan al ejecutar; no se inventan métricas.

**Objetivo:** regresión retrospectiva de `yield_g/planta`; métrica principal RMSE (g/planta). **Precaución:** 32 observaciones y 16 parcelas; se mantiene el holdout por parcela y se evita fuga en selección de variables, preprocesamiento y stacking.


In [ ]:
# Librerías y configuración general
from pathlib import Path
import re, warnings, unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from scipy.stats import spearmanr, skew, shapiro, levene, f_oneway, kruskal
from statsmodels.stats.multitest import multipletests

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 190)
sns.set_theme(style='whitegrid', palette='deep')

SEED = 54
np.random.seed(SEED)

# Detectar raíz del repositorio aunque Jupyter se abra desde notebook/.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'data' / 'raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        'No se encontró data/raw. Abre este notebook desde la raíz del repositorio '
        'o guárdalo dentro de la carpeta notebook/.'
    )
BASE_DIR = ROOT / 'data' / 'raw'
OUT = ROOT / 'outputs' / 'avance3_modelado'
FIG = OUT / 'figuras'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

FILES = {
    'antiox': 'Antiox_data.xlsx',
    'sampling': 'Sampling May 2023.xlsx',
    'harvests': 'harvests.xlsx',
    'soil': 'Dataset_final_suelo.xlsx',
    'leaf': 'Dataset_final_leaf.xlsx',
    'poly_raw': 'alldata_polyphenols.xlsx',
    'anth_raw': 'Anthocyanins.xlsx',
    'vitc_raw': 'VitC_MarinaPerez.xlsx',
    'poly_table': 'Table polyphenols.xlsx',
}

missing_files = [name for name in FILES.values() if not (BASE_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f'Faltan archivos en {BASE_DIR}: ' + ', '.join(missing_files)
    )

print('Raíz del repositorio:', ROOT)
print('Directorio RAW:', BASE_DIR)
print('Archivos originales detectados:', len(FILES))
print('Resultados:', OUT)

In [ ]:
SHEETS = {
    'antiox': {'Antiox_data': None},
    'sampling': {'all_data': None},
    'harvests': {'harvests': None, 'productions_kg': None},
    'soil': {'Final': None},
    'leaf': {'Isotopes': None, 'RGB': None, 'Nitrates': None, 'Nutrients': 1},
    'poly_raw': {'alldata_polyphenols': None, 'data_polyphenols': None},
    'anth_raw': {'Cyanidin gluc': None, 'Pelargonidin gluc': None, 'Pelargonidin rutin': None},
    'vitc_raw': {'Hoja3': None},
}

inventory=[]
for key, name in FILES.items():
    book = pd.ExcelFile(BASE_DIR / name)
    for sheet in book.sheet_names:
        inventory.append({'Fuente':key,'Archivo':name,'Hoja':sheet})
    if key in SHEETS:
        for sheet in SHEETS[key]:
            if sheet not in book.sheet_names:
                raise ValueError(f'{name}: falta hoja {sheet!r}. Hojas disponibles: {book.sheet_names}')

inventory_df = pd.DataFrame(inventory)
raw={}
for key, sheets in SHEETS.items():
    for sheet, header in sheets.items():
        args={'sheet_name':sheet}
        if header is not None:
            args['header']=header
        df=pd.read_excel(BASE_DIR / FILES[key], **args)
        df.columns=[str(c).strip() for c in df.columns]
        raw[(key,sheet)] = df
        inventory_df.loc[(inventory_df.Fuente==key)&(inventory_df.Hoja==sheet),'Filas']=len(df)
        inventory_df.loc[(inventory_df.Fuente==key)&(inventory_df.Hoja==sheet),'Columnas']=df.shape[1]

display(inventory_df.loc[inventory_df['Filas'].notna(), ['Fuente','Archivo','Hoja','Filas','Columnas']])


In [ ]:
antiox_raw = raw[('antiox','Antiox_data')]
fruit_raw = raw[('sampling','all_data')]
harvest_raw = raw[('harvests','harvests')]
yield_raw = raw[('harvests','productions_kg')]
soil_raw = raw[('soil','Final')]
iso_raw = raw[('leaf','Isotopes')]
rgb_raw = raw[('leaf','RGB')]
nitr_raw = raw[('leaf','Nitrates')]
nut_raw = raw[('leaf','Nutrients')].copy()
nut_raw = nut_raw.loc[:, ~nut_raw.columns.astype(str).str.startswith('Unnamed')]
nitr_raw = nitr_raw.loc[nitr_raw['Parcel/Plot'].notna()].copy()

CORE = {
    'Antioxidantes':antiox_raw, 'Calidad fruto':fruit_raw, 'Producción':yield_raw,
    'Cosechas por fecha':harvest_raw, 'Suelo':soil_raw, 'Hoja isótopos':iso_raw,
    'Hoja nitratos':nitr_raw, 'Hoja nutrientes':nut_raw, 'Hoja RGB':rgb_raw
}

def describe_source(name, df):
    return {
        'Fuente':name, 'Filas':len(df), 'Columnas':df.shape[1],
        'Numéricas':df.select_dtypes(include=np.number).shape[1],
        'NA totales':int(df.isna().sum().sum()),
        'Filas idénticas':int(df.duplicated().sum())
    }

source_profile = pd.DataFrame([describe_source(k,d) for k,d in CORE.items()])
display(source_profile)


In [ ]:
# Estandarización de tratamientos.
SOIL_TO_T = {
    'COMPOST':'T1', 'N-RICH':'T2', 'NRICH':'T2',
    'RCW-HD':'T3','RWC-HD':'T3','WR-HD':'T3','RCWHD':'T3',
    'RCW-LD':'T4','RWC-LD':'T4','WR-LD':'T4','RCWLD':'T4'
}
# IMPORTANTE: en todos los gráficos/tablas visibles se usarán nombres, no códigos.
T_TO_NAME = {'T1':'Compost', 'T2':'N-rich', 'T3':'RCW-HD', 'T4':'RCW-LD'}
TREATMENT_ORDER = ['Compost','N-rich','RCW-HD','RCW-LD']

def clean_cv(s):
    v=s.astype('string').str.strip().str.upper()
    return v.map({'DO':'Donna','DONA':'Donna','DONNA':'Donna','DR':'Dream','DREAM':'Dream'})

def clean_code(s):
    return s.astype('string').str.strip().str.upper().str.replace(r'\s+','',regex=True)

def require_cols(df, cols, label):
    missing=set(cols)-set(df.columns)
    if missing:
        raise KeyError(f'{label}: columnas no encontradas: {sorted(missing)}; disponibles={list(df.columns)}')

KEYS=['Cultivar','Treatment','Replicate']

def make_keys(df, cv, tr, rep, label):
    require_cols(df,[cv,tr,rep],label)
    return pd.DataFrame({
        'Cultivar':clean_cv(df[cv]),
        'Treatment':clean_code(df[tr]),
        'Replicate':clean_code(df[rep])
    }, index=df.index)

def check_unique(df, keys, label):
    na=df[keys].isna().any(axis=1)
    dup=df.duplicated(keys,keep=False)
    if na.any() or dup.any():
        display(df.loc[na|dup,keys].head(20))
        raise ValueError(f'{label}: claves nulas o duplicadas; no se realizará unión insegura.')

display(pd.DataFrame([{'Código original':k,'Tratamiento mostrado':v} for k,v in T_TO_NAME.items()]))


In [ ]:
# Construcción de las fuentes principales en Cultivar × Treatment × Replicate.
ant = make_keys(antiox_raw,'cultivar','Treatment','Replicate','Antiox_data')
for c in antiox_raw.columns:
    if c not in ['Sample','cultivar','Treatment','Replicate','New_block']:
        ant[c]=pd.to_numeric(antiox_raw[c],errors='coerce')
check_unique(ant,KEYS,'Antioxidantes')

fruit = make_keys(fruit_raw,'Cv','Treatment','Repl','Sampling/all_data')
for c in ['Weight','Hydrat','Firm','Hue','Chroma','BRIX','Acidity','GA','GGA','CSI']:
    if c in fruit_raw:
        fruit['fruit_'+c]=pd.to_numeric(fruit_raw[c],errors='coerce')
check_unique(fruit,KEYS,'Calidad del fruto')

yld = make_keys(yield_raw,'Cv','Treatment','Repl','Producción')
for c in ['Com Ton/ha','No com Ton/ha','Nº de plantes','g/planta']:
    if c in yield_raw:
        yld['yield_'+c]=pd.to_numeric(yield_raw[c],errors='coerce')
check_unique(yld,KEYS,'Producción')

sets_keys={name:set(map(tuple,df[KEYS].to_numpy())) for name,df in {'Antiox':ant,'Fruto':fruit,'Producción':yld}.items()}
assert sets_keys['Antiox']==sets_keys['Fruto']==sets_keys['Producción'], 'Las claves de las fuentes principales no coinciden.'

# Suelo 2023: una observación por Treatment × Replicate.
require_cols(soil_raw,['Treatment','Block','Year'],'Suelo/Final')
soil23=soil_raw.loc[pd.to_numeric(soil_raw['Year'],errors='coerce').eq(2023)].copy()
soil_t_raw=clean_code(soil23['Treatment']).str.replace('_','-',regex=False)
soil=pd.DataFrame({'Treatment':soil_t_raw.map(SOIL_TO_T), 'Replicate':clean_code(soil23['Block'])}, index=soil23.index)
for c in soil23.columns:
    if c not in ['Treatment','Year','Block','Plot']:
        soil['soil_'+c]=pd.to_numeric(soil23[c],errors='coerce')
if soil.Treatment.isna().any():
    raise ValueError('Existen tratamientos de suelo sin mapear: '+str(soil23.loc[soil.Treatment.isna(),'Treatment'].unique()))
check_unique(soil,['Treatment','Replicate'],'Suelo 2023')

# Hojas: réplicas técnicas en la misma unidad/fecha se resumen por media.
def pivot_leaf(df,cv,tr,rep,time,measures,prefix,label):
    require_cols(df,[cv,tr,rep]+measures,label)
    k=make_keys(df,cv,tr,rep,label)
    k['Time']=pd.Series(time,index=df.index).astype('string').str.strip()
    for c in measures:
        k[c]=pd.to_numeric(df[c],errors='coerce')
    piv=k.pivot_table(index=KEYS, columns='Time', values=measures, aggfunc='mean')
    piv.columns=[f'{prefix}_{str(c)}_{str(t)}' for c,t in piv.columns]
    return piv.reset_index()

iso_t=iso_raw['Date'].astype('string').str.extract('(March|May)',expand=False)
leaf_iso=pivot_leaf(iso_raw,'Cv','Treatment','Replicate',iso_t,
                    [c for c in ['%N','d13C'] if c in iso_raw], 'leaf_iso','Isotopes')

nut_vars=[c for c in nut_raw.columns if c not in ['Sample','Parcel/Plot','Cv','Treatment','Repl','Time']
          and pd.api.types.is_numeric_dtype(nut_raw[c])]
leaf_nut=pivot_leaf(nut_raw,'Cv','Treatment','Repl',nut_raw['Time'],nut_vars,'leaf_nut','Nutrients')
leaf_no3=pivot_leaf(nitr_raw,'Cv','Treatment','Repl',nitr_raw['Time'],
                    [c for c in ['mg NO3-/ g DW','mg NO3-/ 100 mg DW'] if c in nitr_raw],
                    'leaf_no3','Nitrates')
leaf_rgb=pivot_leaf(rgb_raw,'Cv','Treatment','Repl',rgb_raw['Time'],
                    [c for c in ['GA','GGA','CSI'] if c in rgb_raw], 'leaf_rgb','RGB')

# Unión maestra.
ana=ant.merge(fruit,on=KEYS,how='outer',validate='one_to_one',indicator='_source_fruit')
ana=ana.merge(yld,on=KEYS,how='outer',validate='one_to_one',indicator='_source_yield')
ana=ana.merge(soil,on=['Treatment','Replicate'],how='left',validate='many_to_one',indicator='_source_soil')
for x in [leaf_iso,leaf_nut,leaf_no3,leaf_rgb]:
    ana=ana.merge(x,on=KEYS,how='left',validate='one_to_one')

ana['Parcel_ID']=ana['Treatment'].astype(str)+'_'+ana['Replicate'].astype(str)
ana['Treatment_name']=ana['Treatment'].map(T_TO_NAME).fillna('Sin mapear')
ana['Cultivar_Treatment']=ana['Cultivar'].astype(str)+' · '+ana['Treatment_name'].astype(str)

assert not ana[KEYS].duplicated().any(), 'La unión generó llaves duplicadas.'
print('Filas analíticas:',len(ana))
print('Parcelas:',ana.Parcel_ID.nunique())
print('Variables numéricas:',len(ana.select_dtypes(include=np.number).columns))
print('Tratamientos visibles:',sorted(ana.Treatment_name.unique()))

if len(ana)!=32 or ana.Parcel_ID.nunique()!=16:
    warnings.warn('Cobertura distinta de 32 filas / 16 parcelas: revisar fuentes y llaves de unión.')


In [ ]:
# Bloques científicos.
design_cols=['Cultivar','Treatment','Treatment_name','Replicate','Parcel_ID','Cultivar_Treatment']
soil_cols=[c for c in ana if c.startswith('soil_') and pd.api.types.is_numeric_dtype(ana[c])]
leaf_nutrition_cols=[c for c in ana if c.startswith(('leaf_nut_','leaf_iso_','leaf_no3_')) and pd.api.types.is_numeric_dtype(ana[c])]
rgb_cols=[c for c in ana if c.startswith('leaf_rgb_') and pd.api.types.is_numeric_dtype(ana[c])]
production_cols=[c for c in ana if c.startswith('yield_') and pd.api.types.is_numeric_dtype(ana[c])]
rgb_duplicate_outcomes=[c for c in ['fruit_GA','fruit_GGA','fruit_CSI'] if c in ana.columns]
# GA/GGA/CSI del archivo de sampling se consideran indicadores de vegetación/RGB, no calidad fisicoquímica del fruto.
# Además, en el EDA previo se observó que duplican las mediciones RGB de mayo; se excluyen de outcomes para evitar relaciones innecesarias.
physchem_cols=[c for c in ana if c.startswith('fruit_') and pd.api.types.is_numeric_dtype(ana[c]) and c not in rgb_duplicate_outcomes]
nutritional_cols=[c for c in ant.columns if c not in KEYS and c in ana.columns and pd.api.types.is_numeric_dtype(ana[c])]

BLOCKS={
    'Diseño experimental':design_cols,
    'Suelo':soil_cols,
    'Hoja / nutrición':leaf_nutrition_cols,
    'Hoja / RGB-fisiología':rgb_cols,
    'Producción':production_cols,
    'Calidad fisicoquímica':physchem_cols,
    'Calidad nutricional / bioactivos':nutritional_cols,
}
block_summary=pd.DataFrame([{'Bloque':k,'Variables':len(v)} for k,v in BLOCKS.items()])
display(block_summary)

X_raw_cols=soil_cols+leaf_nutrition_cols+rgb_cols
outcome_families={
    'Producción':production_cols,
    'Fisicoquímica':physchem_cols,
    'Nutricional':nutritional_cols,
}

advantages = [
    'Diseño balanceado: 2 cultivares × 4 tratamientos × 4 réplicas.',
    'Múltiples capas del sistema: suelo, hoja, fisiología/RGB, producción y fruto.',
    'Mediciones temporales en suelo y hoja que permiten estudiar evolución.',
    'Resultados químicos y bioactivos suficientes para formular objetivos múltiples.'
]
limitations = [
    'n≈32 es pequeño frente al número de variables (p >> n).',
    'Algunas variables presentan faltantes y distribuciones sesgadas.',
    'Suelo se comparte por parcela; no todas las columnas son observaciones independientes.',
    'Treatment mezcla tipo de enmienda y práctica de labranza; no permite separar ambos efectos causalmente.',
    'Los modelos solo pueden generalizar razonablemente dentro de tratamientos/cultivares similares a los observados.'
]

display(Markdown('### Ventajas de la información\n' + '\n'.join([f'- {x}' for x in advantages])))
display(Markdown('### Limitaciones que condicionan el modelado\n' + '\n'.join([f'- {x}' for x in limitations])))

print('Columnas GA/GGA/CSI excluidas de calidad fisicoquímica por pertenecer a RGB/redundancia:', rgb_duplicate_outcomes)


## 1. Diagnóstico de la tarea y calidad de los datos
**Regresión**, porque el rendimiento en gramos por planta es continuo. La exploración comprueba faltantes, duplicados, valores extremos y correlación entre predictores; los valores atípicos no se eliminan automáticamente. El análisis no pretende estimar causalidad ni realizar pronóstico anterior a la cosecha.

In [ ]:
# Ingeniería temporal: baseline + delta, reutilizando la lógica de Avance 2.
def feature_builder(df):
    z=df.copy(); created={}; removed=[]; original=list(z.columns)

    for family in ['leaf_iso_','leaf_no3_','leaf_nut_']:
        may_cols=[c for c in original if c.startswith(family) and c.endswith('_May')]
        for may in may_cols:
            march=may[:-3]+'March'
            if march in z.columns:
                new='FE_delta_'+may.replace('_May','')
                z[new]=pd.to_numeric(z[may],errors='coerce')-pd.to_numeric(z[march],errors='coerce')
                created[new]=f'Cambio May - March; baseline conservado en {march}.'
                removed.append(may)

    rgb_metrics=sorted({c.rsplit('_',1)[0] for c in original if c.startswith('leaf_rgb_')})
    for base in rgb_metrics:
        dec,mar,may=base+'_December',base+'_March',base+'_May'
        if dec in z and mar in z:
            d1='FE_delta_'+base+'_March_minus_December'
            z[d1]=pd.to_numeric(z[mar],errors='coerce')-pd.to_numeric(z[dec],errors='coerce')
            created[d1]=f'Cambio March - December para {base}; December funciona como baseline.'
            removed.append(mar)
        if mar in z and may in z:
            d2='FE_delta_'+base+'_May_minus_March'
            z[d2]=pd.to_numeric(z[may],errors='coerce')-pd.to_numeric(z[mar],errors='coerce')
            created[d2]=f'Cambio May - March para {base}.'
            removed.append(may)

    removed=list(dict.fromkeys([c for c in removed if c in z.columns]))
    z=z.drop(columns=removed)
    return z,created,removed

ana_fe, derived, temporal_removed = feature_builder(ana)

# Control de disponibilidad temporal (el modo retrospectivo NO equivale a pronostico temprano).
PREDICTION_MODE = 'retrospectivo'  # 'temprano' solo tras verificar calendario de muestreo
CUTOFF_PERIOD = None  # December, March o May si PREDICTION_MODE='temprano'
PERIOD_ORDER = {'December': 0, 'March': 1, 'May': 2}

def latest_measurement_period(column):
    if 'May_minus_March' in column: return 'May'
    if 'March_minus_December' in column: return 'March'
    if column.startswith('FE_delta_leaf_'): return 'May'
    for period in PERIOD_ORDER:
        if column.endswith('_' + period): return period
    return None

if PREDICTION_MODE == 'temprano':
    if CUTOFF_PERIOD not in PERIOD_ORDER:
        raise ValueError('CUTOFF_PERIOD debe ser December, March o May.')
    excluded_temporal = [c for c in ana_fe.columns
                         if latest_measurement_period(c) is not None
                         and PERIOD_ORDER[latest_measurement_period(c)] > PERIOD_ORDER[CUTOFF_PERIOD]]
    ana_fe = ana_fe.drop(columns=excluded_temporal)
    print('Excluidas por corte temporal:', len(excluded_temporal))
    print('Advertencia: revisar tambien fechas reales de suelo y cosecha; el nombre no prueba disponibilidad.')
else:
    excluded_temporal = []
    print('Modo retrospectivo: asociaciones posteriores al cultivo; NO prediccion temprana.')

soil_fe=[c for c in ana_fe if c.startswith('soil_') and pd.api.types.is_numeric_dtype(ana_fe[c])]
leaf_fe=[c for c in ana_fe if (c.startswith(('leaf_nut_','leaf_iso_','leaf_no3_')) or c.startswith(('FE_delta_leaf_nut_','FE_delta_leaf_iso_','FE_delta_leaf_no3_'))) and pd.api.types.is_numeric_dtype(ana_fe[c])]
rgb_fe=[c for c in ana_fe if (c.startswith('leaf_rgb_') or c.startswith('FE_delta_leaf_rgb_')) and pd.api.types.is_numeric_dtype(ana_fe[c])]
X_fe_cols=list(dict.fromkeys(soil_fe+leaf_fe+rgb_fe))

work_cols=list(dict.fromkeys(['Cultivar','Treatment','Treatment_name','Replicate','Parcel_ID','Cultivar_Treatment'] + X_fe_cols + production_cols + physchem_cols + nutritional_cols))
work=ana_fe[work_cols].copy().replace([np.inf,-np.inf],np.nan)

print('Predictores originales suelo+hoja+RGB:',len(X_raw_cols))
print('Predictores después de FE temporal:',len(X_fe_cols))
print('Características delta creadas:',len(derived))
print('Mediciones finales redundantes retiradas:',len(temporal_removed))


In [ ]:
# Split por parcela para evitar que información compartida de suelo aparezca en train y test.
from sklearn.model_selection import GroupShuffleSplit

split=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=SEED)
itr,ite=next(split.split(work, groups=work['Parcel_ID']))
tr=work.iloc[itr].copy(); te=work.iloc[ite].copy()

assert set(tr.Parcel_ID).isdisjoint(set(te.Parcel_ID))
print('Entrenamiento:',tr.shape,'| Prueba:',te.shape)
print('Parcelas train/test:',tr.Parcel_ID.nunique(),te.Parcel_ID.nunique())
print('Overlap de parcelas:',set(tr.Parcel_ID)&set(te.Parcel_ID))
print('Parcelas test:',sorted(te.Parcel_ID.unique()))


In [ ]:
TARGET = 'yield_g/planta'
assert TARGET in work.columns, f'Objetivo ausente: {TARGET}'
assert not work[['Cultivar','Treatment','Replicate']].duplicated().any()
assert set(tr.Parcel_ID).isdisjoint(set(te.Parcel_ID))
PRED_NUM = list(X_fe_cols)
PRED_CAT = ['Treatment_name', 'Cultivar']
PREDICTORS = PRED_NUM + PRED_CAT
assert TARGET not in PREDICTORS
train_data = tr.loc[tr[TARGET].notna()].copy()
test_data = te.loc[te[TARGET].notna()].copy()
print('Tarea: regresión | objetivo:', TARGET)
print('Train/test:', len(train_data), len(test_data), ' | parcelas:',train_data.Parcel_ID.nunique(),test_data.Parcel_ID.nunique())
print('Predictores numéricos:',len(PRED_NUM),' | categóricos:',PRED_CAT)
print('Faltantes en X train:',int(train_data[PREDICTORS].isna().sum().sum()))
print('Duplicados de clave:',int(work.duplicated(['Cultivar','Treatment','Replicate']).sum()))
print('Objetivo (train):');display(train_data[TARGET].describe().to_frame())
fig,ax=plt.subplots(figsize=(7,3));ax.hist(train_data[TARGET],bins=8,edgecolor='white');ax.set(xlabel='Rendimiento (g/planta)',ylabel='Frecuencia',title='Distribución del objetivo en entrenamiento');plt.tight_layout();plt.show()
# Auditoría descriptiva de atípicos (sin eliminarlos).
q1,q3=train_data[TARGET].quantile([.25,.75]);iqr=q3-q1
print('Atípicos IQR (objetivo, train):',int(((train_data[TARGET]<q1-1.5*iqr)|(train_data[TARGET]>q3+1.5*iqr)).sum()))

## 2. Diseño de evaluación sin fuga de información

- **Train/test**: `GroupShuffleSplit` de la fase anterior, agrupando por parcela; la prueba se consulta solo tras elegir el modelo final.
- **Comparación inicial**: `GroupKFold(4)` en entrenamiento, con exactamente los mismos folds para los seis algoritmos.
- **Selección supervisada de variables**: dentro del `Pipeline`, en cada fold; cuotas científicas de Top 12 (4 suelo, 6 nutrición, 2 RGB). La selección de la fase anterior no se reutiliza como ranking fijo.
- **Preprocesamiento**: imputación, estandarización y OHE dentro del fold. No se emplean variables de producción, calidad del fruto, identificadores o réplicas como predictores.
- **Cautela**: las mediciones de mayo solo justifican interpretación retrospectiva salvo verificación cronológica adicional.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import f_regression
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.exceptions import ConvergenceWarning
from time import perf_counter
import warnings

# Modificables; no cambiar después de consultar test.
CV_OUTER_FOLDS = 4
CV_INNER_FOLDS = 3
FEATURE_QUOTAS = (4,6,2)
HOLDOUT_EVAL = True  # ejecutar UNA sola vez después de fijar el diseño.

class ScientificSelector(BaseEstimator, TransformerMixin):
    def __init__(self, quotas=(4,6,2), numeric_cols=None, cat_cols=None):
        self.quotas=quotas;self.numeric_cols=numeric_cols;self.cat_cols=cat_cols
    def fit(self,X,y):
        selected=[]
        for prefix,k in zip(('soil','leaf','rgb'),self.quotas):
            if prefix=='soil':cols=[c for c in self.numeric_cols if c.startswith('soil_')]
            elif prefix=='rgb':cols=[c for c in self.numeric_cols if c.startswith(('leaf_rgb_','FE_delta_leaf_rgb_'))]
            else:cols=[c for c in self.numeric_cols if c.startswith(('leaf_','FE_delta_leaf_')) and not c.startswith(('leaf_rgb_','FE_delta_leaf_rgb_'))]
            cols=[c for c in cols if c in X.columns and X[c].notna().any()]
            if not cols or k==0:continue
            z=X[cols].apply(pd.to_numeric,errors='coerce');z=z.fillna(z.median())
            valid=[c for c in cols if z[c].notna().all() and z[c].nunique()>1]
            if not valid:continue
            with warnings.catch_warnings():
                warnings.simplefilter('ignore');scores,_=f_regression(z[valid],np.asarray(y).ravel())
            rank=pd.DataFrame({'feature':valid,'F':np.nan_to_num(scores,nan=-1,posinf=-1,neginf=-1)}).sort_values(['F','feature'],ascending=[False,True])
            selected+=rank.head(k).feature.tolist()
        self.selected_=list(dict.fromkeys(selected))
        if not self.selected_:raise ValueError('Sin predictores numéricos seleccionados.')
        return self
    def transform(self,X):return X.loc[:,self.selected_+self.cat_cols].copy()

def one_hot():
    try:return OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False)
    except TypeError:return OneHotEncoder(drop='first',handle_unknown='ignore',sparse=False)

def pipeline_for(estimator):
    prep=ColumnTransformer([
        ('numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),make_column_selector(dtype_include=np.number)),
        ('categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('ohe',one_hot())]),PRED_CAT)
    ],sparse_threshold=0)
    return Pipeline([('selector',ScientificSelector(quotas=FEATURE_QUOTAS,numeric_cols=PRED_NUM,cat_cols=PRED_CAT)),('prep',prep),('model',estimator)])

MODELS = {
 'Regresión lineal':LinearRegression(),
 'Ridge':Ridge(alpha=10.0),
 'Lasso':Lasso(alpha=1.0,max_iter=30000,random_state=SEED),
 'SVR':SVR(kernel='rbf',C=1.0,epsilon=0.1, gamma='scale'),
 'KNN':KNeighborsRegressor(n_neighbors=3,weights='uniform'),
 'Árbol de decisión':DecisionTreeRegressor(max_depth=2,min_samples_leaf=3,random_state=SEED),
}
X=train_data[PREDICTORS];y=train_data[TARGET].astype(float);groups=train_data.Parcel_ID
outer_cv=GroupKFold(n_splits=min(CV_OUTER_FOLDS,groups.nunique()))
scoring={'rmse':'neg_root_mean_squared_error','mae':'neg_mean_absolute_error','r2':'r2'}
print('Algoritmos individuales:',list(MODELS))

## Ensambles adicionales sin alterar el Avance 4

Se mantienen **sin cambios** los datos, partición por parcela, semilla, preprocesamiento, selección científica, algoritmos individuales y sus espacios de hiperparámetros del Avance 4. Se agregan cinco ensambles: Bagging, Random Forest, Extra Trees (homogéneos) y Voting, Stacking (heterogéneos). KNN y Ridge se reoptimizan con **las mismas grillas exactas** del Avance 4 en cada partición externa. Los ensambles Voting/Stacking incluyen KNN y Ridge; su configuración fija se usa solo como referencia inicial, y no equivale a trasladar todos los hiperparámetros óptimos de la fase anterior. La selección se realiza por RMSE de validación agrupada, no por test.

In [ ]:
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, ExtraTreesRegressor, VotingRegressor
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import GroupKFold
from time import perf_counter
import inspect

# Wrapper de stacking con CV por grupos: usa las mismas parcelas en el ajuste del meta-modelo.
class GroupStackingRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, knn_neighbors=3, ridge_alpha=10., tree_depth=2, meta_alpha=10., n_splits=3):
        self.knn_neighbors=knn_neighbors; self.ridge_alpha=ridge_alpha
        self.tree_depth=tree_depth; self.meta_alpha=meta_alpha; self.n_splits=n_splits
    def _models(self):
        return [pipeline_for(clone(MODELS['KNN']).set_params(n_neighbors=self.knn_neighbors)),
                pipeline_for(clone(MODELS['Ridge']).set_params(alpha=self.ridge_alpha)),
                pipeline_for(DecisionTreeRegressor(max_depth=self.tree_depth,min_samples_leaf=3,random_state=SEED))]
    def fit(self,X,y,groups=None):
        if groups is None: raise ValueError('Stacking requiere groups=Parcel_ID para evitar fuga.')
        X=X.reset_index(drop=True); y=pd.Series(np.asarray(y)).reset_index(drop=True)
        g=pd.Series(np.asarray(groups)).reset_index(drop=True)
        folds=GroupKFold(n_splits=min(self.n_splits,g.nunique()))
        oof=np.full((len(X),3),np.nan)
        for a,b in folds.split(X,y,g):
            for j,est in enumerate(self._models()):
                est.fit(X.iloc[a],y.iloc[a]);oof[b,j]=est.predict(X.iloc[b])
        assert np.isfinite(oof).all()
        self.meta_=Ridge(alpha=self.meta_alpha).fit(oof,y)
        self.base_=self._models()
        for est in self.base_:est.fit(X,y)
        return self
    def predict(self,X):
        return self.meta_.predict(np.column_stack([m.predict(X) for m in self.base_]))

def make_bagging(n_estimators=50,max_samples=.8,min_samples_leaf=3):
    kw={'n_estimators':n_estimators,'max_samples':max_samples,'random_state':SEED,'n_jobs':1}
    base=DecisionTreeRegressor(max_depth=3,min_samples_leaf=min_samples_leaf,random_state=SEED)
    if 'estimator' in inspect.signature(BaggingRegressor).parameters:kw['estimator']=base
    else:kw['base_estimator']=base
    return BaggingRegressor(**kw)

def candidate(name,params):
    if name=='KNN individual':return pipeline_for(clone(MODELS['KNN']))
    if name=='Ridge individual':return pipeline_for(clone(MODELS['Ridge']))
    if name=='Bagging':return pipeline_for(make_bagging(**params))
    if name=='Random Forest':return pipeline_for(RandomForestRegressor(random_state=SEED,n_jobs=1,**params))
    if name=='Extra Trees':return pipeline_for(ExtraTreesRegressor(random_state=SEED,n_jobs=1,**params))
    if name=='Voting':
        return VotingRegressor([('knn',pipeline_for(clone(MODELS['KNN']).set_params(n_neighbors=params['knn_neighbors']))),
                                ('ridge',pipeline_for(clone(MODELS['Ridge']).set_params(alpha=params['ridge_alpha']))),
                                ('tree',pipeline_for(DecisionTreeRegressor(max_depth=2,min_samples_leaf=3,random_state=SEED)))],weights=[2,2,1])
    if name=='Stacking':return GroupStackingRegressor(**params)
    raise KeyError(name)

SPACES={
 'KNN individual':{},
 'Ridge individual':{},
 'Bagging':{'n_estimators':[30,70],'max_samples':[.75,1.],'min_samples_leaf':[2,3]},
 'Random Forest':{'n_estimators':[50,100],'max_depth':[2,4],'min_samples_leaf':[2,3],'max_features':[.7]},
 'Extra Trees':{'n_estimators':[50,100],'max_depth':[2,4],'min_samples_leaf':[2,3],'max_features':[.7]},
 'Voting':{'knn_neighbors':[3],'ridge_alpha':[10.]},
 'Stacking':{'knn_neighbors':[3],'ridge_alpha':[10.],'meta_alpha':[10.,100.],'tree_depth':[2],'n_splits':[3]}
}
print('Modelos:',list(SPACES))
print('Configuraciones:',{k:len(list(ParameterGrid(v))) for k,v in SPACES.items()})


## 2. Optimización y evaluación anidada por parcela
Para cada modelo y cada fold externo, se optimiza RMSE mediante `GroupKFold(3)` interno. **Para stacking**, la CV interna y la construcción de meta-características respetan las parcelas. La tabla principal utiliza métricas externas comparables; el tiempo incluye búsqueda interna y ajuste del candidato elegido.


In [ ]:
# Reutilizar SIN MODIFICACIONES las grillas del Avance 4 para KNN y Ridge.
PARAM_GRIDS = {
 'KNN': {'model__n_neighbors':[2,3,4,5], 'model__weights':['uniform','distance'], 'model__p':[1,2], 'selector__quotas':[(3,3,2),(4,6,2)]},
 'Ridge': {'model__alpha':[0.1,1,3,10,30,100,300,1000], 'selector__quotas':[(3,3,2),(4,6,2),(5,8,3),(6,10,4)]},
}

def fit_candidate(name,params,xa,ya,ga):
    est=candidate(name,params)
    if name=='Stacking':est.fit(xa,ya,groups=ga)
    else:est.fit(xa,ya)
    return est

def tune(name,xa,ya,ga):
    if name in ('KNN individual','Ridge individual'):
        model_name=name.replace(' individual','')
        search=GridSearchCV(pipeline_for(clone(MODELS[model_name])), PARAM_GRIDS[model_name],
                            scoring='neg_root_mean_squared_error', cv=GroupKFold(n_splits=CV_INNER_FOLDS),
                            n_jobs=1,refit=True,error_score='raise')
        search.fit(xa,ya,groups=ga)
        return search.best_estimator_,search.best_params_,-search.best_score_
    cv=GroupKFold(n_splits=3)
    best_score=np.inf;best_params=None
    for p in ParameterGrid(SPACES[name]):
        losses=[]
        for a,b in cv.split(xa,ya,ga):
            fit=fit_candidate(name,p,xa.iloc[a],ya.iloc[a],ga.iloc[a])
            pred=fit.predict(xa.iloc[b]);losses.append(np.sqrt(mean_squared_error(ya.iloc[b],pred)))
        score=np.mean(losses)
        if score<best_score:best_score=score;best_params=p
    fitted=fit_candidate(name,best_params,xa,ya,ga)
    return fitted,best_params,best_score

results=[];fold_rows=[];oof_predictions={}
for name in SPACES:
    truths=np.full(len(X),np.nan);preds=np.full(len(X),np.nan)
    fold_rmse=[];fold_mae=[];fold_r2=[];elapsed=0.
    for fold,(a,b) in enumerate(outer_cv.split(X,y,groups),1):
        start=perf_counter()
        fitted,params,inner=tune(name,X.iloc[a],y.iloc[a],groups.iloc[a])
        elapsed+=perf_counter()-start
        pp=np.asarray(fitted.predict(X.iloc[b])).ravel()
        truths[b]=y.iloc[b];preds[b]=pp
        rmse=np.sqrt(mean_squared_error(y.iloc[b],pp));mae=mean_absolute_error(y.iloc[b],pp)
        r2=r2_score(y.iloc[b],pp)
        fold_rmse.append(rmse);fold_mae.append(mae);fold_r2.append(r2)
        fold_rows.append({'Modelo':name,'Fold':fold,'RMSE':rmse,'MAE':mae,'R2':r2,'Parametros':str(params)})
    oof_predictions[name]=(truths,preds)
    results.append({'Modelo':name,'Tipo':'Individual' if 'individual' in name else ('Heterogéneo' if name in ('Voting','Stacking') else 'Homogéneo'),
                    'RMSE_CV':np.mean(fold_rmse),'RMSE_sd':np.std(fold_rmse,ddof=1),
                    'MAE_CV':np.mean(fold_mae),'R2_CV':np.mean(fold_r2),'Tiempo_entrenamiento_s':elapsed})
comparison5=pd.DataFrame(results).sort_values(['RMSE_CV','RMSE_sd']).reset_index(drop=True)
comparison5.insert(0,'Posición',np.arange(1,len(comparison5)+1))
display(comparison5.round(3))
OUT5=OUT/'avance5_ensambles';OUT5.mkdir(parents=True,exist_ok=True)
comparison5.to_csv(OUT5/'comparativa_modelos.csv',index=False)
pd.DataFrame(fold_rows).to_csv(OUT5/'resultados_por_fold.csv',index=False)


## Selección del modelo final

La tabla incluye KNN y Ridge con las **grillas originales del Avance 4** y cinco ensambles nuevos. Orden: RMSE CV (menor es mejor); métricas adicionales MAE, R² y desviación del RMSE, además del tiempo de entrenamiento. Los valores de prueba se reportan únicamente para el candidato elegido. Si el ganador cambia respecto del Avance 4, ello no implica un error: es una comparación de candidatos nuevos bajo el mismo diseño de validación. **Limitación:** los hiperparámetros de los modelos base de Voting y Stacking se fijan en valores iniciales del Avance 4; no son automáticamente las configuraciones optimizadas ganadoras de aquella fase.

In [ ]:
winner=comparison5.iloc[0]['Modelo']
start=perf_counter()
final5,final_params,inner_rmse=tune(winner,X,y,groups)
final_fit_time=perf_counter()-start
Xt=test_data[PREDICTORS];yt=test_data[TARGET].astype(float)
yhat=np.asarray(final5.predict(Xt)).ravel()
final_metrics={'Modelo':winner,'RMSE_test':np.sqrt(mean_squared_error(yt,yhat)),
               'MAE_test':mean_absolute_error(yt,yhat),'R2_test':r2_score(yt,yhat),
               'Tiempo_entrenamiento_final_s':final_fit_time,'n_test':len(yt),
               'parcelas_test':test_data.Parcel_ID.nunique()}
print('MODELO ELEGIDO:',winner,'| Parámetros:',final_params)
print('Motivo: mínimo RMSE CV externo; comprobar estabilidad (RMSE_sd), MAE, R2 y costo.')
display(pd.DataFrame([final_metrics]).round(3))
pd.DataFrame([final_metrics]).to_csv(OUT5/'evaluacion_final.csv',index=False)


## 4. Cuatro gráficos del modelo final con interpretación
Se muestran: (1) RMSE por modelo, (2) observado frente a predicho, (3) residuos frente a predicción y (4) distribución de errores absolutos; además, (5) importancia por permutación sobre test como análisis exploratorio. **ROC, precisión-recall y matriz de confusión no aplican** a una regresión continua.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

fig,ax=plt.subplots(figsize=(9,4));ax.barh(comparison5.Modelo,comparison5.RMSE_CV,xerr=comparison5.RMSE_sd,capsize=3);ax.invert_yaxis()
ax.set(xlabel='RMSE CV ± desviación entre folds (g/planta)',title='Gráfico 1 · Comparación de modelos');plt.tight_layout();plt.savefig(OUT5/'01_comparacion.png',dpi=170);plt.show()
print('Interpretación 1: el primer modelo presenta el menor RMSE promedio; barras de error grandes advierten inestabilidad entre parcelas. Revisar si la diferencia es sustantiva frente a KNN y Ridge.')

fig,ax=plt.subplots(figsize=(6,5));ax.scatter(yt,yhat);lims=[min(yt.min(),yhat.min()),max(yt.max(),yhat.max())];ax.plot(lims,lims,'--',color='gray');ax.set(xlabel='Observado (g/planta)',ylabel='Predicho (g/planta)',title='Gráfico 2 · Observado vs predicho');plt.tight_layout();plt.savefig(OUT5/'02_observado_predicho.png',dpi=170);plt.show()
print('Interpretación 2: puntos cercanos a la diagonal indican buen ajuste; separaciones verticales representan errores. Revisar si se subestiman rendimientos altos o sobreestiman bajos.')

resid=np.asarray(yt)-yhat
fig,ax=plt.subplots(figsize=(6,4));ax.scatter(yhat,resid);ax.axhline(0,color='gray',ls='--');ax.set(xlabel='Predicho (g/planta)',ylabel='Residuo observado - predicho',title='Gráfico 3 · Residuos');plt.tight_layout();plt.savefig(OUT5/'03_residuos.png',dpi=170);plt.show()
print('Interpretación 3: residuos sin tendencia ni patrón de abanico son deseables; valores positivos implican subestimación. Con ocho muestras, no se puede concluir normalidad ni homocedasticidad.')

fig,ax=plt.subplots(figsize=(7,4));ax.bar(np.arange(len(yt)),np.abs(resid));ax.axhline(np.mean(np.abs(resid)),color='gray',ls='--',label='MAE test');ax.legend();ax.set(xlabel='Observación de prueba',ylabel='Error absoluto (g/planta)',title='Gráfico 4 · Error por observación');plt.tight_layout();plt.savefig(OUT5/'04_errores_absolutos.png',dpi=170);plt.show()
print('Interpretación 4: las barras revelan si el MAE se debe a pocos casos extremos o a errores generalizados; analizar parcelas con errores altos antes de proponer uso operativo.')

# Permutación por columnas en test: exploratorio y potencialmente inestable con n=8.
imp=permutation_importance(final5,Xt,yt,n_repeats=15,random_state=SEED,scoring='neg_root_mean_squared_error',n_jobs=1)
importance=pd.Series(imp.importances_mean,index=Xt.columns).sort_values(ascending=False).head(12)
fig,ax=plt.subplots(figsize=(9,5));ax.barh(importance.index[::-1],importance.values[::-1]);ax.set(xlabel='Incremento de RMSE por permutación (g/planta)',title='Gráfico 5 · Importancia exploratoria de variables');plt.tight_layout();plt.savefig(OUT5/'05_importancia.png',dpi=170);plt.show()
print('Interpretación 5: valores positivos indican mayor deterioro predictivo al alterar una variable. Las correlaciones y el test pequeño vuelven esta importancia inestable y NO causal.')


## Conclusiones y criterios de la rúbrica

- **Ensambles (60 puntos):** cinco modelos (tres homogéneos, dos heterogéneos). Para una correspondencia estricta con la rúbrica, Voting/Stacking deben usar los **modelos optimizados finales** del Avance 4, no solo los mismos algoritmos; se deja identificada esta mejora pendiente.
- **Selección (20 puntos):** tabla ordenada por RMSE CV con KNN/Ridge, MAE, R² y tiempos; justificar modelo por generalización, estabilidad y costo, sin seleccionar por test.
- **Gráficos (20 puntos):** visualizar e interpretar al menos cuatro resultados del modelo ganador. Con 32 observaciones, evitar afirmaciones de superioridad estadística.

Los resultados del notebook deben recalcularse tras estos cambios; las salidas guardadas de la versión previa no son válidas.